# 02 — Bronze para Silver

**Arquivo:** 02-bronze-para-silver.ipynb
**Objetivo:** integrar os informes trimestrais de 2020 a 2026, aplicar regras de qualidade e derivar variáveis
**Entrada:** dados/bronze/cvm-inf-trimestral/*.zip
**Saída:** dados/silver/imoveis/ (Parquet) e dados/quarentena/imoveis/ (Parquet)
**Onde roda:** local (.venv) — **Camada:** Bronze → Silver

In [9]:
# Ferramentas: zipfile abre os ZIP sem extrair, pandas manipula as tabelas,
# Path monta caminhos de forma segura no Windows e na nuvem.
import zipfile
import pandas as pd
from pathlib import Path

# Caminhos centralizados em variáveis. Na migração para a nuvem, só estas linhas mudam.
RAIZ = Path("..").resolve()
BRONZE_TRI = RAIZ / "dados" / "bronze" / "cvm-inf-trimestral"
SILVER = RAIZ / "dados" / "silver"
QUARENTENA = RAIZ / "dados" / "quarentena"

print("Bronze existe?", BRONZE_TRI.exists())

Bronze existe? True


In [10]:
# Tabela de-para: nome antigo (arquivo de 2020) → nome atual (2021 em diante).
# Fica fora da função porque é uma regra do projeto, não um passo da carga.
MAPA_COLUNAS = {
    'CNPJ_Fundo': 'CNPJ_Fundo_Classe',
    'Nome_Fundo': 'Nome_Fundo_Classe',
}

def carregar_imoveis_logisticos(ano):
    # Monta o nome do ZIP a partir do ano recebido.
    arquivo_zip = BRONZE_TRI / f"inf_trimestral_fii_{ano}.zip"

    # Lê as duas tabelas necessárias de dentro do mesmo ZIP, sem extrair nada.
    with zipfile.ZipFile(arquivo_zip) as z:
        with z.open(f"inf_trimestral_fii_geral_{ano}.csv") as f:
            df_geral = pd.read_csv(f, sep=';', encoding='latin-1')
        with z.open(f"inf_trimestral_fii_imovel_{ano}.csv") as f:
            df_imovel = pd.read_csv(f, sep=';', encoding='latin-1')

    # Harmoniza o esquema: renomeia colunas antigas para os nomes atuais.
    # Em anos que já usam o nome novo, o rename não encontra nada e não altera nada.
    df_geral = df_geral.rename(columns=MAPA_COLUNAS)
    df_imovel = df_imovel.rename(columns=MAPA_COLUNAS)

    # Seleciona os CNPJs dos fundos logísticos DAQUELE ano (evita viés de sobrevivência).
    filtro = df_geral['Segmento_Atuacao'] == 'Logística'
    cnpjs = df_geral.loc[filtro, 'CNPJ_Fundo_Classe'].unique()

    # Mantém só os imóveis desses fundos, numa tabela independente.
    df = df_imovel[df_imovel['CNPJ_Fundo_Classe'].isin(cnpjs)].copy()

    # Marca cada linha com o ano de origem — será a chave do particionamento.
    df['ano'] = ano

    return df

In [11]:
# Executa a função passando 2025. O resultado fica guardado em df_2025.
df_2025 = carregar_imoveis_logisticos(2025)

# Confere se reproduz exatamente o número encontrado no notebook 01.
print("Formato:", df_2025.shape)
print("Bate com o notebook 01 (1640)?", len(df_2025) == 1640)

# Verifica o tipo de cada coluna, sem assumir. Area e os percentuais devem ser float64.
print("\nTipos das colunas-chave:")
print(df_2025[['Area', 'Percentual_Vacancia', 'Percentual_Receitas_FII', 'ano']].dtypes)

Formato: (1640, 20)
Bate com o notebook 01 (1640)? True

Tipos das colunas-chave:
Area                       float64
Percentual_Vacancia        float64
Percentual_Receitas_FII    float64
ano                          int64
dtype: object


In [12]:
# Função auxiliar: lê só o cabeçalho de um CSV (nrows=0 não carrega nenhuma linha
# de dados) e devolve a lista de nomes de colunas. É rápido e não pesa na memória.
def listar_colunas(ano, tabela):
    arquivo_zip = BRONZE_TRI / f"inf_trimestral_fii_{ano}.zip"
    with zipfile.ZipFile(arquivo_zip) as z:
        with z.open(f"inf_trimestral_fii_{tabela}_{ano}.csv") as f:
            return pd.read_csv(f, sep=';', encoding='latin-1', nrows=0).columns.tolist()

# Compara 2020 com 2025. set() transforma a lista em conjunto, o que permite
# subtrair: "o que existe em um e não existe no outro".
for tabela in ['geral', 'imovel']:
    c2020 = set(listar_colunas(2020, tabela))
    c2025 = set(listar_colunas(2025, tabela))
    print(f"=== Tabela {tabela} ===")
    print("Só em 2020:", sorted(c2020 - c2025))
    print("Só em 2025:", sorted(c2025 - c2020))
    print()

# Descobre em que ano o nome mudou, verificando a presença da coluna ano a ano.
print("=== A coluna CNPJ_Fundo_Classe existe em cada ano? ===")
for ano in range(2020, 2027):
    print(ano, "→ geral:", 'CNPJ_Fundo_Classe' in listar_colunas(ano, 'geral'),
               "| imovel:", 'CNPJ_Fundo_Classe' in listar_colunas(ano, 'imovel'))

=== Tabela geral ===
Só em 2020: ['CNPJ_Fundo', 'Nome_Fundo']
Só em 2025: ['CNPJ_Fundo_Classe', 'Nome_Fundo_Classe', 'Tipo_Fundo_Classe']

=== Tabela imovel ===
Só em 2020: ['CNPJ_Fundo']
Só em 2025: ['CNPJ_Fundo_Classe']

=== A coluna CNPJ_Fundo_Classe existe em cada ano? ===
2020 → geral: False | imovel: False
2021 → geral: True | imovel: True
2022 → geral: True | imovel: True
2023 → geral: True | imovel: True
2024 → geral: True | imovel: True
2025 → geral: True | imovel: True
2026 → geral: True | imovel: True


In [15]:
# range(2020, 2027) gera os anos de 2020 a 2026. O número final NÃO entra na
# sequência — por isso 2027, para que 2026 seja incluído.
anos = range(2020, 2027)

# Lista vazia que vai receber uma tabela por ano.
tabelas = []

# Para cada ano: carrega, mostra quantos imóveis vieram e guarda na lista.
# O print antes da carga mostra qual ano estava sendo processado se algo quebrar.
for ano in anos:
    print(f"Carregando {ano}...", end=" ")
    df_ano = carregar_imoveis_logisticos(ano)
    print(f"{len(df_ano)} registros (imóvel × trimestre)")
    tabelas.append(df_ano)

# Empilha todas as tabelas da lista numa só. ignore_index=True renumera as linhas
# de 0 até o fim, em vez de repetir a numeração de cada ano.
df_todos = pd.concat(tabelas, ignore_index=True)

print("\nTotal consolidado:", df_todos.shape)

Carregando 2020... 469 registros (imóvel × trimestre)
Carregando 2021... 628 registros (imóvel × trimestre)
Carregando 2022... 1027 registros (imóvel × trimestre)
Carregando 2023... 902 registros (imóvel × trimestre)
Carregando 2024... 956 registros (imóvel × trimestre)
Carregando 2025... 1640 registros (imóvel × trimestre)
Carregando 2026... 570 registros (imóvel × trimestre)

Total consolidado: (6192, 20)


In [16]:
# Chave que identifica um imóvel num trimestre: fundo + data + nome do imóvel.
# Se essa combinação aparecer mais de uma vez, há duplicidade.
chave = ['CNPJ_Fundo_Classe', 'Data_Referencia', 'Nome_Imovel']

# keep=False marca TODAS as linhas repetidas, não só a segunda ocorrência.
duplicados = df_todos.duplicated(subset=chave, keep=False)
print("Registros envolvidos em duplicidade:", duplicados.sum())

# Conta quantos registros existem de cada versão do informe.
print("\nDistribuição da coluna Versao:")
print(df_todos['Versao'].value_counts().sort_index())

# Se houver duplicados, mostra alguns para inspeção visual.
if duplicados.sum() > 0:
    print("\nExemplos de duplicidade:")
    print(df_todos.loc[duplicados, chave + ['Versao', 'Area']].sort_values(chave).head(10).to_string())

Registros envolvidos em duplicidade: 181

Distribuição da coluna Versao:
Versao
1    5609
2     458
3     125
Name: count, dtype: int64

Exemplos de duplicidade:
       CNPJ_Fundo_Classe Data_Referencia Nome_Imovel  Versao      Area
4166  14.217.108/0001-45      2025-03-31     Terreno       1  36575.00
4167  14.217.108/0001-45      2025-03-31     Terreno       1   2100.00
4168  14.217.108/0001-45      2025-03-31     Terreno       1  14587.09
4169  14.217.108/0001-45      2025-03-31     Terreno       1  33650.00
4170  14.217.108/0001-45      2025-03-31     Terreno       1   2516.30
4171  14.217.108/0001-45      2025-03-31     Terreno       1  30367.10
4172  14.217.108/0001-45      2025-03-31     Terreno       1  10635.13
4174  14.217.108/0001-45      2025-03-31     Terreno       1  16784.49
4175  14.217.108/0001-45      2025-06-30     Terreno       1  33650.00
4176  14.217.108/0001-45      2025-06-30     Terreno       1  30367.10


In [17]:
# 1. Quais classes de imóvel existem? Define quais registros fazem sentido para vacância.
print("=== Valores da coluna Classe ===")
print(df_todos['Classe'].value_counts())

# 2. Algum informe (fundo + trimestre) aparece com mais de uma versão no arquivo?
# groupby agrupa como o GROUP BY do SQL; nunique conta valores distintos, como COUNT(DISTINCT).
versoes_por_informe = df_todos.groupby(['CNPJ_Fundo_Classe', 'Data_Referencia'])['Versao'].nunique()
print("\n=== Versões por informe ===")
print("Total de informes (fundo + trimestre):", len(versoes_por_informe))
print("Informes com mais de uma versão no arquivo:", (versoes_por_informe > 1).sum())

# 3. Duplicata verdadeira: linhas idênticas em TODAS as colunas.
# Sem o parâmetro subset, o duplicated compara a linha inteira.
print("\n=== Duplicatas exatas ===")
print("Linhas idênticas em todas as colunas:", df_todos.duplicated().sum())

=== Valores da coluna Classe ===
Classe
Imóveis para renda acabados         5737
Imóveis para renda em construção     198
Imóveis para venda acabados          195
Imóveis para venda em construção      62
Name: count, dtype: int64

=== Versões por informe ===
Total de informes (fundo + trimestre): 1008
Informes com mais de uma versão no arquivo: 0

=== Duplicatas exatas ===
Linhas idênticas em todas as colunas: 0


In [18]:
# Limite de plausibilidade definido no notebook 01 (DA-16).
LIMITE_AREA = 1_000_000

# ---- Filtro de escopo ----
# Vacância só faz sentido para imóvel pronto e destinado a aluguel.
# Os demais tipos saem do estudo, mas NÃO vão para a quarentena: não estão errados.
CLASSE_ESTUDO = 'Imóveis para renda acabados'
df = df_todos[df_todos['Classe'] == CLASSE_ESTUDO].copy()
print(f"Registros no escopo: {len(df)} de {len(df_todos)}")

# ---- Regras de qualidade ----
# Começa com o motivo vazio. Cada regra violada acrescenta seu código ao texto,
# separado por ponto e vírgula. Um registro pode violar mais de uma regra.
df['motivo_quarentena'] = ''

df.loc[df['Area'].isna(), 'motivo_quarentena'] += 'area_ausente;'
df.loc[df['Area'] == 0, 'motivo_quarentena'] += 'area_zero;'
df.loc[df['Area'] > LIMITE_AREA, 'motivo_quarentena'] += 'area_acima_limite;'
df.loc[df['Percentual_Vacancia'].isna(), 'motivo_quarentena'] += 'vacancia_ausente;'
df.loc[(df['Percentual_Vacancia'] < 0) | (df['Percentual_Vacancia'] > 1),
       'motivo_quarentena'] += 'vacancia_fora_faixa;'

# ---- Separação ----
# Motivo vazio = aprovado. Qualquer motivo preenchido = quarentena.
em_quarentena = df['motivo_quarentena'] != ''
df_silver = df[~em_quarentena].drop(columns='motivo_quarentena')
df_quarentena = df[em_quarentena]

print(f"Aprovados:   {len(df_silver)}")
print(f"Quarentena:  {len(df_quarentena)} ({len(df_quarentena) / len(df) * 100:.1f}%)")

print("\nMotivos da quarentena:")
print(df_quarentena['motivo_quarentena'].value_counts())

# Conferência: sobrou algum registro com "Terreno" no nome entre os aprovados?
qtd_terreno = df_silver['Nome_Imovel'].str.contains('terreno', case=False, na=False).sum()
print(f"\nAprovados com 'terreno' no nome: {qtd_terreno}")

Registros no escopo: 5737 de 6192
Aprovados:   5607
Quarentena:  130 (2.3%)

Motivos da quarentena:
motivo_quarentena
area_zero;            106
area_acima_limite;     23
vacancia_ausente;       1
Name: count, dtype: int64

Aprovados com 'terreno' no nome: 0


In [19]:
# Converte a data de texto para o tipo data. Permite ordenar corretamente e
# extrair partes da data, como o trimestre.
df_silver['Data_Referencia'] = pd.to_datetime(df_silver['Data_Referencia'])
df_silver['Trimestre'] = df_silver['Data_Referencia'].dt.quarter

# ---- Variáveis derivadas (feature engineering) ----
# Ocupação por complementaridade: Percentual_Locado tem 97,7% de vazios (DA-14).
df_silver['Taxa_Ocupacao'] = 1 - df_silver['Percentual_Vacancia']

# Conversão de medida relativa em absoluta: m² podem ser somados entre imóveis
# de tamanhos diferentes; percentuais não.
df_silver['Area_Vaga_m2'] = df_silver['Area'] * df_silver['Percentual_Vacancia']
df_silver['Area_Locada_m2'] = df_silver['Area'] * df_silver['Taxa_Ocupacao']

# ---- Seleção de colunas ----
# Mantém só o que tem uso na análise. Colunas de construção e venda pertencem às
# classes excluídas por escopo; Percentual_Locado é inutilizável.
COLUNAS_SILVER = [
    'CNPJ_Fundo_Classe', 'Data_Referencia', 'ano', 'Trimestre', 'Versao',
    'Nome_Imovel', 'Endereco', 'Classe', 'Area', 'Numero_Unidades',
    'Percentual_Vacancia', 'Taxa_Ocupacao', 'Area_Vaga_m2', 'Area_Locada_m2',
    'Percentual_Inadimplencia', 'Percentual_Receitas_FII',
    'Percentual_Imovel_Total_Investido',
]
df_silver = df_silver[COLUNAS_SILVER]

# ---- Teste de sanidade ----
# Área vaga + área locada tem que reconstituir a área total, em todas as linhas.
# .abs().max() pega a maior diferença encontrada; deve ser praticamente zero.
diferenca = (df_silver['Area_Vaga_m2'] + df_silver['Area_Locada_m2'] - df_silver['Area']).abs().max()
print("Maior diferença entre vaga + locada e área total:", diferenca)

print("\nFormato da Silver:", df_silver.shape)
df_silver.head()

Maior diferença entre vaga + locada e área total: 5.820766091346741e-11

Formato da Silver: (5607, 17)


,CNPJ_Fundo_Classe,Data_Referencia,ano,Trimestre,Versao,Nome_Imovel,Endereco,Classe,Area,Numero_Unidades,Percentual_Vacancia,Taxa_Ocupacao,Area_Vaga_m2,Area_Locada_m2,Percentual_Inadimplencia,Percentual_Receitas_FII,Percentual_Imovel_Total_Investido
0,02.508.235/0001-14,2020-03-31,2020,1,1,Contagem,"Samambaia, Perobas e Campo Do Meio - Contagem ...",Imóveis para renda acabados,348156.92,2.0,0.143614,0.856386,50000.207909,298156.712091,0.0,0.998300,NaN
1,02.508.235/0001-14,2020-06-30,2020,2,1,Contagem,"Samambaia, Perobas e Campo Do Meio - Contagem ...",Imóveis para renda acabados,348156.92,2.0,0.143614,0.856386,50000.207909,298156.712091,0.0,0.997800,NaN
2,02.508.235/0001-14,2020-09-30,2020,3,1,Contagem,"Samambaia, Perobas e Campo Do Meio - Contagem ...",Imóveis para renda acabados,348156.92,2.0,0.143614,0.856386,50000.207909,298156.712091,0.0,0.999556,NaN
3,02.508.235/0001-14,2020-12-31,2020,4,1,Contagem,"Samambaia, Perobas e Campo Do Meio - Contagem ...",Imóveis para renda acabados,348156.92,2.0,0.143614,0.856386,50000.207909,298156.712091,0.0,0.998284,NaN
5,05.437.916/0001-27,2020-03-31,2020,1,1,Torre de Elevadores - Interlagos - SP,"Av. Interlagos, 4.455",Imóveis para renda acabados,3298.00,1.0,0.000000,1.000000,0.000000,3298.000000,0.0,0.125018,NaN


In [20]:
import shutil

DESTINO_SILVER = SILVER / "imoveis"
DESTINO_QUARENTENA = QUARENTENA / "imoveis.parquet"

# Idempotência: apaga a gravação anterior antes de gravar de novo.
# Sem isso, rodar a célula duas vezes duplicaria os dados dentro das partições.
if DESTINO_SILVER.exists():
    shutil.rmtree(DESTINO_SILVER)

# Grava a Silver particionada por ano: cria uma subpasta ano=AAAA para cada ano (DA-09).
df_silver.to_parquet(DESTINO_SILVER, partition_cols=['ano'], index=False)

# Quarentena num arquivo único, com o motivo de cada rejeição. Não é consultada
# com frequência, então não precisa de partição.
df_quarentena.to_parquet(DESTINO_QUARENTENA, index=False)

# ---- Verificação: lê de volta o que foi gravado ----
df_conferencia = pd.read_parquet(DESTINO_SILVER)
print("Registros gravados na Silver:", len(df_conferencia))
print("Bate com o que estava na memória?", len(df_conferencia) == len(df_silver))

print("\nPartições criadas:")
for pasta in sorted(DESTINO_SILVER.iterdir()):
    print(" ", pasta.name)

Registros gravados na Silver: 5607
Bate com o que estava na memória? True

Partições criadas:
  ano=2020
  ano=2021
  ano=2022
  ano=2023
  ano=2024
  ano=2025
  ano=2026


## Conclusões da camada Silver

- Carga dos informes trimestrais de 2020 a 2026 por meio de função única, aplicada ano a ano.
- Fundos logísticos selecionados pela tabela `geral` de cada ano, evitando viés de sobrevivência (DA-17).
- Esquema harmonizado: o arquivo de 2020 usa `CNPJ_Fundo` e `Nome_Fundo`; renomeados para os nomes atuais (DA-18).
- Sem versões concorrentes nem duplicatas exatas na tabela `imovel`: deduplicação não necessária.
- Escopo: apenas "Imóveis para renda acabados" — 5.737 de 6.192 registros (DA-19).
- Qualidade: 130 registros em quarentena (2,3%) — 106 área zero, 23 área acima de 1 milhão de m², 1 vacância ausente (DA-20).
- A ausência de vacância era estrutural: concentrada nas classes fora do escopo.
- Variáveis derivadas: `Taxa_Ocupacao`, `Area_Vaga_m2`, `Area_Locada_m2`, `Trimestre` (DA-21).
- Saída: 5.607 registros e 17 colunas em Parquet particionado por ano; gravação idempotente (DA-22).